# Naive RAG with AWS Textract

**Pipeline Overview:**
1. Upload PDF to S3
2. Extract text + tables via AWS Textract (async)
3. Chunk and embed extracted content with OpenAI
4. Store embeddings in FAISS vector store
5. Query with GPT-4o using retrieved context

## 1. Imports & Configuration

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import time
import base64

import boto3
import fitz                                         # PyMuPDF — PDF rendering
from trp import Document as TextractDocument        # Parses Textract block JSON

from langchain_community.docstore.document import Document as LCDocument
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

In [2]:
# --- Configuration ---
# Set your credentials and resource names here

os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
S3_BUCKET = os.getenv('S3_BUCKET')
S3_PATH = "aws-textract-input"
AWS_REGION=os.getenv('REGION')
LOCAL_PDF_PATH="../data/docling_report-3-5.pdf"

# Boto3 clients
session  = boto3.Session(region_name=AWS_REGION)
s3       = session.client("s3")
textract = session.client("textract")

## 2. Upload PDF to S3

In [6]:
# Textract's async API requires the document to live in S3
s3_key = f"{S3_PATH}/{os.path.basename(LOCAL_PDF_PATH)}"
s3.upload_file(LOCAL_PDF_PATH, S3_BUCKET, s3_key)
print(f"Uploaded '{LOCAL_PDF_PATH}' → s3://{S3_BUCKET}/{s3_key}")

Uploaded '../data/docling_report-3-5.pdf' → s3://msaifee-experiment-bucket-958614257254-us-east-1-an/aws-textract-input/docling_report-3-5.pdf


## 3. AWS Textract — Async Document Analysis

In [ ]:
def run_textract(bucket: str, key: str) -> list[dict]:
    """
    Start an async Textract job for a PDF stored in S3.
    Polls until completion, then paginates through all result pages.

    Returns a list of raw Textract response dicts (one per paginated call).
    """
    # Start the job — TABLES gives structured table blocks, LAYOUT gives figure/text regions
    job = textract.start_document_analysis(
        DocumentLocation={"S3Object": {"Bucket": bucket, "Name": key}},
        FeatureTypes=["TABLES", "LAYOUT"]
    )
    job_id = job["JobId"]
    print(f"Textract job started: {job_id}")

    # Poll until the job finishes
    while True:
        resp   = textract.get_document_analysis(JobId=job_id)
        status = resp["JobStatus"]
        if status in ("SUCCEEDED", "FAILED"):
            break
        print("  ... waiting")
        time.sleep(5)

    if status == "FAILED":
        raise RuntimeError("Textract job failed.")

    # Collect all paginated pages of results
    pages = [resp]
    token = resp.get("NextToken")
    while token:
        resp  = textract.get_document_analysis(JobId=job_id, NextToken=token)
        pages.append(resp)
        token = resp.get("NextToken")

    print(f"Textract completed — {len(pages)} page(s) of results.")
    return pages

In [8]:
raw_results = run_textract(S3_BUCKET, s3_key)

Textract job started: 40310933d63db97fe59ec8474e7917ed71b416de42b18ab64e25964cbd39ce4d
  ... waiting
  ... waiting
Textract completed — 2 page(s) of results.


In [10]:
raw_results

[{'DocumentMetadata': {'Pages': 3},
  'JobStatus': 'SUCCEEDED',
  'NextToken': 'TOu6MPCTh6dMvQ5woJZtsv6qFPcf5sEdwq7dC6zKVLXpmlt2weO802AShysIqNWoSESCUFd5A67ihaFFuvak6t0yaNrd4a9GXY+iAdt/bH+qzJEWEc8TxdAOn3J5CBac6dYCRZU=',
  'Blocks': [{'BlockType': 'PAGE',
    'Geometry': {'BoundingBox': {'Width': 1.0,
      'Height': 1.0,
      'Left': 0.0,
      'Top': 0.0},
     'Polygon': [{'X': 0.0, 'Y': 0.0},
      {'X': 1.0, 'Y': 0.0},
      {'X': 1.0, 'Y': 1.0},
      {'X': 0.0, 'Y': 1.0}]},
    'Id': 'cfc29aa4-1589-42a5-8f6a-4e0bc0ed24e1',
    'Relationships': [{'Type': 'CHILD',
      'Ids': ['38fac044-a290-4c03-8c9b-12ebf1a0ceaf',
       '340aa3dd-472c-41de-9ccb-9875b8255240',
       '1c8975ce-9c9e-418f-98a1-db87c9759851',
       '84cde942-eea3-463e-afb9-29273bd6fa5f',
       'cd2a438f-e448-4c17-b6e4-7e6437dffc96',
       '8790f6a0-c0a3-4168-9afe-9de55cfbee7d',
       'a5f532b8-5949-4c80-b3f4-8871b1a33b6d',
       '2317213a-cbe0-4791-b73e-3aa5864c8a5f',
       'f7cc0880-847c-49cb-bc80-dcf7ae346f

## 4. Parse Textract Output → LangChain Documents

In [28]:
def crop_figure(pdf_bytes: bytes, page_num: int, bbox: dict, dpi: int = 200) -> str:
    """
    Crop a figure region from a PDF page and return it as a base64-encoded PNG.

    Textract bounding boxes use normalized [0, 1] coordinates;
    we convert them to PDF point units using the page dimensions.
    """
    doc  = fitz.open(stream=pdf_bytes, filetype="pdf")
    page = doc.load_page(page_num - 1)          # Textract pages are 1-indexed
    w, h = page.rect.width, page.rect.height

    rect = fitz.Rect(
        bbox["Left"]                   * w,
        bbox["Top"]                    * h,
        (bbox["Left"] + bbox["Width"]) * w,
        (bbox["Top"]  + bbox["Height"]) * h,
    )
    pix = page.get_pixmap(clip=rect, dpi=dpi)
    return base64.b64encode(pix.tobytes("png")).decode()


def parse_textract(textract_pages: list[dict], pdf_path: str) -> list[LCDocument]:
    """
    Converts Textract block output into typed LangChain Documents:
      - 'table'  : Markdown-formatted table text
      - 'image'  : placeholder text + base64 PNG in metadata
      - 'text'   : plain paragraph text
    """
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    # Flatten blocks from all paginated responses
    all_blocks = [b for page in textract_pages for b in page["Blocks"]]
    block_map  = {b["Id"]: b for b in all_blocks}

    docs = []

    # --- Tables (via TRP helper) ---
    trp_doc = TextractDocument({"Blocks": all_blocks})

    # Track page number per table using the raw block's PAGE blocks
    page_blocks = [b for b in all_blocks if b["BlockType"] == "PAGE"]
    page_index = {b["Id"]: b.get("Page", i+1) for i, b in enumerate(page_blocks)}

    for i, pg in enumerate(trp_doc.pages):
        page_num = i + 1  # TRP pages are ordered; use 1-based index directly

        for table in pg.tables:
            rows  = []
            cells = list(table.rows[0].cells)

            for idx, row in enumerate(table.rows):
                row_text = "| " + " | ".join(c.text.strip() for c in row.cells) + " |"
                rows.append(row_text)
                if idx == 0:
                    rows.append("| " + " | ".join(["---"] * len(cells)) + " |")

            if rows:
                docs.append(LCDocument(
                    page_content="TABLE:\n" + "\n".join(rows),
                    metadata={"modality": "table", "page": page_num}
                ))

    # --- Figures and plain text (from LAYOUT blocks) ---
    for block in all_blocks:
        b_type   = block["BlockType"]
        page_num = block.get("Page", 1)
        bbox     = block["Geometry"]["BoundingBox"]

        if b_type == "LAYOUT_FIGURE":
            img_b64 = crop_figure(pdf_bytes, page_num, bbox)
            
            # Generate caption for the figure using GPT model
            llm = ChatOpenAI(model="gpt-5.4-mini", max_completion_tokens=256)
            caption_resp = llm.invoke([{
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}", "detail": "high"}},
                    {"type": "text", "text": "Describe this figure or diagram concisely in 2-3 sentences."}
                ]
            }])
            caption = caption_resp.content
            
            docs.append(LCDocument(
                page_content=f"[Figure Caption]: {caption}",
                metadata={"modality": "image", "page": page_num, "image_base64": img_b64}
            ))

        elif b_type == "LAYOUT_TEXT" and "Relationships" in block:
            # Reconstruct paragraph text from child LINE blocks
            child_ids = block["Relationships"][0]["Ids"]
            text = " ".join(
                block_map[cid].get("Text", "")
                for cid in child_ids
                if block_map.get(cid, {}).get("BlockType") == "LINE"
            ).strip()
            if text:
                docs.append(LCDocument(
                    page_content=text,
                    metadata={"modality": "text", "page": page_num}
                ))

    print(f"Extracted {len(docs)} chunks ({sum(1 for d in docs if d.metadata['modality']=='table')} tables, "
          f"{sum(1 for d in docs if d.metadata['modality']=='image')} images, "
          f"{sum(1 for d in docs if d.metadata['modality']=='text')} text blocks)")
    return docs

In [29]:
chunks = parse_textract(raw_results, LOCAL_PDF_PATH)

Extracted 25 chunks (1 tables, 1 images, 23 text blocks)


In [30]:
chunks

[Document(metadata={'modality': 'table', 'page': 3}, page_content='TABLE:\n| CPU | Thread |  | native backend |  |  | pypdfium | backend |\n| --- | --- | --- | --- | --- | --- | --- | --- |\n|  | budget | TTS | Pages/s | Mem | TTS | Pages/s | Mem |\n| Apple M3 Max (16 cores) | 4 16 | 177 S 167 S | 1.27 1.34 | 6.20 GB | 103 S 92 S | 2.18 2.45 | 2.56 GB |\n| Intel(R) Xeon E5-2690 (16 cores) | 4 16 | 375 S 244 S | 0.60 0.92 | 6.16 GB | 239 S 143 S | 0.94 1.57 | 2.42 GB |'),
 Document(metadata={'modality': 'image', 'page': 1, 'image_base64': '...'}, page_content='[Figure Caption]: The diagram shows a PDF-processing pipeline that starts with parsing PDF pages, then feeds them into a model pipeline for OCR, layout analysis, and table structure extraction. The extracted results are then assembled and post-processed before being serialized into JSON or Markdown.'),
 Document(metadata={'modality': 'text', 'page': 1}, page_content="Figure 1: Sketch of Docling's default processing pipeline. The i

## 5. Build FAISS Vector Store

In [31]:
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embed_model)
print("Vector store built.")

Vector store built.


## 7. Query — Naive RAG

In [38]:
def query_rag(question: str, store: FAISS, k: int = 5) -> str:
    """
    Naive RAG: retrieve top-k chunks, build a multimodal prompt.
    """
    docs = store.similarity_search(question, k=k)

    # Build the user message content list
    content = [{"type": "text", "text": f"Question: {question}\n\nContext:"}]

    for doc in docs:
        modality = doc.metadata["modality"]

        if modality == "figure":
            # Attach the cropped figure as a vision input
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{doc.metadata['image_b64']}",
                    "detail": "high"
                }
            })
        else:
            content.append({
                "type": "text",
                "text": f"[{modality.upper()}] {doc.page_content}"
            })

    messages = [
        {"role": "system", "content": "Answer the user's question using only the provided context (text, tables, figures)."},
        {"role": "user",   "content": content}
    ]

    llm    = ChatOpenAI(model="gpt-5.4-mini", max_completion_tokens=1024)
    answer = llm.invoke(messages).content
    
    # Format source references from retrieved chunks
    sources = [
        f"Page {d.metadata['page']} [{d.metadata['modality'].upper()}]: {d.page_content[:100]}..."
        for d in docs
    ]
    return {"answer": answer, "sources": sources}


In [40]:
query = "Sketch of Docling’s default processing pipeline"

result = query_rag(query, vector_store, 5)
print("--- ANSWER ---")
print(result["answer"])
print("\n--- SOURCES ---")
for s in result["sources"]:
    print(s)


--- ANSWER ---
Docling’s default processing pipeline can be sketched as:

1. **Parse PDF pages**
   - The input PDF is first parsed into pages.

2. **Feed page images into the model pipeline**
   - Pages are processed as **images at 72 dpi**.
   - The model pipeline is the customizable core of the system.

3. **Run page-level prediction models**
   - The pipeline performs tasks such as:
     - **OCR**
     - **layout analysis**
     - **table structure extraction**

4. **Post-process predictions**
   - Bounding-box proposals are filtered to remove overlaps based on **confidence and size**.
   - Results are intersected with PDF text tokens to form complete units like:
     - paragraphs
     - section titles
     - list items
     - captions
     - figures
     - tables

5. **Assemble document output**
   - All page-level results are merged into a structured document object.

6. **Final post-processing**
   - The document is further enhanced by:
     - detecting the **document language**